# Chapter 3
## Essential Operations

This is a basic how-to guide following the contents of Chapter 3 of *Delta Lake: The Definitive Guide*

Consider it a quick field guide to running all your CRUD operations.

In [ ]:
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, DATASETS, WORK, set_conf, tree, rm, upload_dir
CH = f"{WORK}/ch03"  # replaces the Docker image's /opt/spark/work-dir

# Start clean so the notebook can be re-run (the Docker image started with an empty work-dir)
rm(CH)
spark.sql("drop table if exists exampleDB.countries2")

# Unique per run: once countries.parquet is converted to Delta below, serverless remembers that path as a
# Delta table even after it is deleted, so writing parquet to the same path again fails (DELTA_INVALID_FORMAT)
import time
countries_parquet = f"{CH}/countries_{int(time.time())}.parquet"

In [2]:
# Create a database for the examples
spark.sql("create database if not exists exampleDB")

DataFrame[]

In [ ]:
# Create some data
data = [
(1, "United Kingdom", "London"),
(2, "Canada", "Toronto")
]

# Create a schema
schema = ["id", "country", "capital"]

# Create a dataframe
sdf = spark.createDataFrame(data, schema=schema)

# Save as a csv to use later
(sdf
 .toPandas()
 .to_csv("countries.csv", header=None)
)

# Save as a paruet to use later
sdf.write.format("parquet").mode("overwrite").save(countries_parquet)

In [4]:
# Make sure the table doesn't already exist
spark.sql("""
DROP TABLE if exists exampleDB.countries
""")

DataFrame[]

In [5]:
# Create an empty table
spark.sql("""
CREATE TABLE exampleDB.countries (
  id LONG,
  country STRING,
  capital STRING
) USING DELTA;
""")

DataFrame[]

In [ ]:
print(CH)

In [ ]:
# Create a table in the work volume so it is inspectable.
# Unity Catalog doesn't allow CREATE TABLE ... LOCATION on a volume, so instead of exampleDB.countries_loc
# this is a path-based table, addressed as delta.`<path>`
spark.sql(f"""
CREATE TABLE delta.`{CH}/countries.delta` (
  id LONG,
  country STRING,
  capital STRING
) USING DELTA
""")

In [8]:
# Basic INSERT INTO example
spark.sql("""
INSERT INTO exampleDB.countries VALUES
(1, 'United Kingdom', 'London'),
(2, 'Canada', 'Toronto')
""")

DataFrame[]

In [ ]:
# INSERT INTO from a parquet file 
spark.sql(f"""
INSERT INTO exampleDB.countries
SELECT * FROM parquet.`{countries_parquet}`;
""")

In [11]:
# Simple CTAS example
spark.sql("""
CREATE TABLE exampleDB.countries2 as
select * from exampleDB.countries
""")

DataFrame[]

In [ ]:
# CONVERT TO DELTA example
spark.sql(f"""
CONVERT TO DELTA parquet.`{countries_parquet}`
""")

#### Check out the _delta_log and view the contents of the JSON file(s)

In [ ]:
tree(f"{CH}/countries.delta/_delta_log")

In [ ]:
import pandas as pd

# *.json only: the .crc files Databricks writes alongside have a clashing schema
spark.read.json(f"{CH}/countries.delta/_delta_log/*.json").show(truncate=False)

In [14]:
# Check the current contents of the table
spark.table("exampleDB.countries").show()

+---+--------------+-------+
| id|       country|capital|
+---+--------------+-------+
|  1|United Kingdom| London|
|  1|United Kingdom| London|
|  2|        Canada|Toronto|
|  2|        Canada|Toronto|
+---+--------------+-------+



In [15]:
# Reading from the table with SQL
spark.sql("""
SELECT
    id,
    capital
FROM
    exampleDB.countries
""")

DataFrame[id: bigint, capital: string]

In [16]:
# Same select statement but with Pyspark
(spark.table("exampleDB.countries")
 .select("id", "capital")
)

DataFrame[id: bigint, capital: string]

In [17]:
# Time travel based on version
spark.sql("""
SELECT DISTINCT id FROM exampleDB.countries VERSION AS OF 1
""").show()

+---+
| id|
+---+
|  1|
|  2|
+---+



In [20]:
# Time travel based on timestamp, you have to be "in bounds" so you would need to adjust to a valid timestamp for this
# It cannot predate or postdate the available versions of the table so we commented out te time tr
spark.sql("""
SELECT count(1) FROM exampleDB.countries -- |TIMESTAMP AS OF "2024-04-20"
""").show()

+--------+
|count(1)|
+--------+
|       4|
+--------+



In [ ]:
# Overwrite with values from another source
spark.sql(f"""
INSERT OVERWRITE TABLE delta.`{CH}/countries.delta`
select distinct * from exampleDB.countries
""")

In [25]:
# Update specific values
spark.sql("""
UPDATE exampleDB.countries
SET country = 'U.K.'
WHERE id = 1;
""")

DataFrame[num_affected_rows: bigint]

In [26]:
# Update values that don't exist (safe but does nothing, see later upsert example for alternative)
spark.sql("""
UPDATE exampleDB.countries
SET country = 'U.S.', capital = 'Washington, D.C.'
WHERE id = 3;
""")

DataFrame[num_affected_rows: bigint]

In [27]:
# DELETE without specifying filters, deletes everything
spark.sql("""
DELETE FROM exampleDB.countries
""")

DataFrame[num_affected_rows: bigint]

In [29]:
# Show everything was deleted
spark.sql("""
select * from exampleDB.countries
""").show()

+---+-------+-------+
| id|country|capital|
+---+-------+-------+
+---+-------+-------+



In [30]:
# INSERT OVERWRITE, if there were any values already they would get deleted too
spark.sql("""
INSERT OVERWRITE exampleDB.countries
VALUES (3, 'U.S.', 'Washington, D.C.');
""")

DataFrame[]

In [ ]:
# Delete with a filtering condition
spark.sql("""
-- The book compares id with '2024-04-20'; Databricks runs in ANSI mode, where that cast is an error
-- (CAST_INVALID_INPUT) instead of silently matching nothing. Same effect with a valid filter:
DELETE FROM exampleDB.countries where id < 2
""")

In [32]:
# Show results
spark.sql("""
select * from exampleDB.countries
""").show()

+---+-------+----------------+
| id|country|         capital|
+---+-------+----------------+
|  3|   U.S.|Washington, D.C.|
+---+-------+----------------+



#### Upsert
Insert records into the table if they don't exist or update them to correspond if they do.

In [ ]:
spark.sql(f"""
MERGE INTO exampleDB.countries A
-- countries.parquet was converted to Delta above; Databricks won't read a Delta directory as parquet
USING (select * from delta.`{countries_parquet}`) B
ON A.id = B.id
WHEN MATCHED THEN
  UPDATE SET
	id = A.id,
	country = B.country,
	capital = B.capital
WHEN NOT MATCHED
  THEN INSERT (
	id,
	country,
	capital
  )
  VALUES (
	B.id,
	B.country,
	B.capital
  )

""")

In [34]:
# Show results
spark.sql("""
select * from exampleDB.countries
""").show()

+---+--------------+----------------+
| id|       country|         capital|
+---+--------------+----------------+
|  1|United Kingdom|          London|
|  3|          U.S.|Washington, D.C.|
|  2|        Canada|         Toronto|
+---+--------------+----------------+

